## Analyzing Common Crawl Data with RDDs

In [1]:
from pyspark.sql import SparkSession

In [2]:
# Create a new SparkSession
spark = SparkSession.builder.getOrCreate()

# Get SparkContext
sc = spark.sparkContext

In [4]:
# Read Domains CSV File into an RDD
common_crawl_domain_counts = sc.textFile('./cc-main-limited-domains.csv')

In [5]:
# Display first few domains from the RDD
common_crawl_domain_counts.take(10)

['367855\t172-in-addr\tarpa\t1',
 '367856\taddr\tarpa\t1',
 '367857\tamphic\tarpa\t1',
 '367858\tbeta\tarpa\t1',
 '367859\tcallic\tarpa\t1',
 '367860\tch\tarpa\t1',
 '367861\td\tarpa\t1',
 '367862\thome\tarpa\t7',
 '367863\tiana\tarpa\t1',
 '367907\tlocal\tarpa\t1']

In [6]:
def fmt_domain_graph_entry(entry):
    """
    Formats a Common Crawl domain graph entry. Extracts the site_id, 
    top-level domain (tld), domain name, and subdomain count as seperate items.
    """
    # Split the entry on delimiter ('\t') into site_id, domain, tld, and num_subdomains
    site_id, domain, tld, num_subdomains = entry.split('\t')        
    return int(site_id), domain, tld, int(num_subdomains)

In [8]:
# Apply `fmt_domain_graph_entry` to the raw data RDD
formatted_host_counts = common_crawl_domain_counts.map(fmt_domain_graph_entry)

# Display the first few entries of the new RDD
formatted_host_counts.take(10)

[(367855, '172-in-addr', 'arpa', 1),
 (367856, 'addr', 'arpa', 1),
 (367857, 'amphic', 'arpa', 1),
 (367858, 'beta', 'arpa', 1),
 (367859, 'callic', 'arpa', 1),
 (367860, 'ch', 'arpa', 1),
 (367861, 'd', 'arpa', 1),
 (367862, 'home', 'arpa', 7),
 (367863, 'iana', 'arpa', 1),
 (367907, 'local', 'arpa', 1)]

In [ ]:
def extract_subdomain_counts(entry):
    """
    Extract the subdomain count from a Common Crawl domain graph entry.
    """    
    # Split the entry on delimiter ('\t') into site_id, domain, tld, and num_subdomains
    site_id, domain, tld, num_subdomains = entry.split('\t')
    
    # return ONLY the num_subdomains
    return int(num_subdomains)

# Apply `extract_subdomain_counts` to the raw data RDD
host_counts = common_crawl_domain_counts.map(extract_subdomain_counts)

In [10]:
# Display the first few entries
host_counts.take(10)

[1, 1, 1, 1, 1, 1, 1, 7, 1, 1]

In [12]:
# Reduce the RDD to a single value, the sum of subdomains, with a lambda function
total_host_counts = host_counts.reduce(lambda a, b: a + b)

# Display result count
total_host_counts

595466

In [14]:
# Stop the sparkContext and the SparkSession
spark.stop()

### Exploring Domain Counts with PySpark DataFrames and SQL

In [5]:
# Create a new SparkSession
spark = SparkSession.builder.getOrCreate()

In [6]:
# Read the target file into a DataFrame
common_crawl = spark.read.option('delimiter', "\t") \
        .option('inferSchema', True) \
        .csv('cc-main-limited-domains.csv')

# Display the DataFrame to the notebook
common_crawl.show(5, truncate=False)

+------+-----------+----+---+
|_c0   |_c1        |_c2 |_c3|
+------+-----------+----+---+
|367855|172-in-addr|arpa|1  |
|367856|addr       |arpa|1  |
|367857|amphic     |arpa|1  |
|367858|beta       |arpa|1  |
|367859|callic     |arpa|1  |
+------+-----------+----+---+
only showing top 5 rows


In [7]:
# Rename the DataFrame's columns with `withColumnRenamed()`
common_crawl = common_crawl.withColumnRenamed('_c0', 'site_id') \
            .withColumnRenamed('_c1', 'domain') \
            .withColumnRenamed('_c2', 'top_level_domain') \
            .withColumnRenamed('_c3', 'num_subdomains')
  
# Display the first few rows of the DataFrame and the new schema
common_crawl.show(5, truncate=False)
common_crawl.printSchema()

+-------+-----------+----------------+--------------+
|site_id|domain     |top_level_domain|num_subdomains|
+-------+-----------+----------------+--------------+
|367855 |172-in-addr|arpa            |1             |
|367856 |addr       |arpa            |1             |
|367857 |amphic     |arpa            |1             |
|367858 |beta       |arpa            |1             |
|367859 |callic     |arpa            |1             |
+-------+-----------+----------------+--------------+
only showing top 5 rows
root
 |-- site_id: integer (nullable = true)
 |-- domain: string (nullable = true)
 |-- top_level_domain: string (nullable = true)
 |-- num_subdomains: integer (nullable = true)



### Reading and Writing Datasets to Disk

In [8]:
# Save the `common_crawl` DataFrame to a series of parquet files
common_crawl.write.parquet('./results/common_crawl/', mode='overwrite')

In [9]:
# Read from parquet directory
common_crawl_domains = spark.read.parquet('./results/common_crawl/')

# Display the first few rows of the DataFrame and the schema
common_crawl_domains.show(5, truncate=False)
common_crawl_domains.printSchema()

+-------+-----------+----------------+--------------+
|site_id|domain     |top_level_domain|num_subdomains|
+-------+-----------+----------------+--------------+
|367855 |172-in-addr|arpa            |1             |
|367856 |addr       |arpa            |1             |
|367857 |amphic     |arpa            |1             |
|367858 |beta       |arpa            |1             |
|367859 |callic     |arpa            |1             |
+-------+-----------+----------------+--------------+
only showing top 5 rows
root
 |-- site_id: integer (nullable = true)
 |-- domain: string (nullable = true)
 |-- top_level_domain: string (nullable = true)
 |-- num_subdomains: integer (nullable = true)



### Querying Domain Counts with PySpark DataFrames and SQL

In [10]:
# Create a temporary view in the metadata for this `SparkSession`
common_crawl_domains.createOrReplaceTempView('crawl')

Calculate the total number of domains for each top-level domain in the dataset.

In [11]:
common_crawl_domains.groupby('top_level_domain').count()\
    .orderBy('count', ascending=False).show(10, truncate=False)

+----------------+-----+
|top_level_domain|count|
+----------------+-----+
|edu             |18547|
|gov             |15007|
|travel          |6313 |
|coop            |5319 |
|jobs            |3893 |
|post            |117  |
|map             |34   |
|arpa            |11   |
+----------------+-----+



Calculate the total number of subdomains for each top-level domain in the dataset.

In [12]:
spark.sql("""SELECT top_level_domain, SUM(num_subdomains)
            FROM crawl
            GROUP BY top_level_domain
            ORDER BY SUM(num_subdomains) DESC""").show(truncate=False)

+----------------+-------------------+
|top_level_domain|sum(num_subdomains)|
+----------------+-------------------+
|edu             |484438             |
|gov             |85354              |
|travel          |10768              |
|coop            |8683               |
|jobs            |6023               |
|post            |143                |
|map             |40                 |
|arpa            |17                 |
+----------------+-------------------+



How many sub-domains does `nps.gov` have? Filter the dataset to that website's entry, display the columns `top_level_domain`, `domain`, and `num_subdomains`.

In [18]:
# Filter the DataFrame using DataFrame Methods
common_crawl_domains.select(['top_level_domain', 'domain', 'num_subdomains']) \
    .filter(common_crawl_domains.domain=='nps') \
    .filter(common_crawl_domains.top_level_domain=='gov') \
    .show(10, truncate=False)

+----------------+------+--------------+
|top_level_domain|domain|num_subdomains|
+----------------+------+--------------+
|gov             |nps   |178           |
+----------------+------+--------------+



In [19]:
# Stop the notebook's `SparkSession` and `sparkContext`
spark.stop()